# Scraping Ulasan Google Play - Bank Jago

Notebook ini melakukan scraping ulasan aplikasi Bank Jago dari Google Play Store.
Output: `data/raw/reviews.csv` (~10.000 baris).

In [9]:
import csv
import os
import time
from google_play_scraper import reviews_all, Sort

In [10]:
APP_ID = "com.jago.digitalBanking"
OUTPUT_FILE = "../data/raw/reviews.csv"
TARGET_COUNT = 10000
LANG = "id"
COUNTRY = "id"

In [11]:
os.makedirs('../data/raw', exist_ok=True)

In [12]:
print(f"Scraping {TARGET_COUNT} reviews for {APP_ID}...")
result = reviews_all(APP_ID, sleep_milliseconds=1000, lang=LANG, country=COUNTRY, sort=Sort.NEWEST)
print(f"Fetched {len(result)} reviews total")

Scraping 10000 reviews for com.jago.digitalBanking...
Fetched 70186 reviews total


In [13]:
# Deduplikasi berdasarkan reviewId
id_filter = set()
unique = []
for r in result:
    if r['reviewId'] not in id_filter:
        id_filter.add(r['reviewId'])
        unique.append(r)
print(f"Unique reviews: {len(unique)}")

Unique reviews: 70186


In [14]:
# Simpan ke CSV (sampel pertama TARGET_COUNT)
sample = unique[:TARGET_COUNT]
with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['reviewId', 'content', 'score', 'at', 'userName'])
    writer.writeheader()
    for r in sample:
        writer.writerow({
            'reviewId': r['reviewId'],
            'content': r['content'],
            'score': r['score'],
            'at': r['at'].isoformat() if hasattr(r['at'], 'isoformat') else r['at'],
            'userName': r['userName'],
        })
print(f'Saved {len(sample)} reviews to {OUTPUT_FILE}')

Saved 10000 reviews to ../data/raw/reviews.csv


In [15]:
import pandas as pd
df = pd.read_csv(OUTPUT_FILE)
print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nScore distribution:')
print(df['score'].value_counts().sort_index())
print(f"\nPositive: {(df['score']>=4).sum()}, Neutral: {(df['score']==3).sum()}, Negative: {(df['score']<=2).sum()}")
df.head()

Dataset: 10000 rows, 5 columns

Score distribution:
score
1    2363
2     290
3     312
4     421
5    6614
Name: count, dtype: int64

Positive: 7035, Neutral: 312, Negative: 2653


,reviewId,content,score,at,userName
0,7f93a1ad-bed3-413e-8590-e1157c2d1fa2,Terlalu banyak cengkonek ni apk mau masuk akun...,1,2026-05-27T02:41:18,Pengguna Google
1,0c97258c-d40b-4c24-811c-59a77187e95f,saya ingin memulihkan akun namun sangat sulit ...,1,2026-05-27T02:23:21,Pengguna Google
2,9c98ad93-9417-4752-84e8-cf8483d538f2,Saya Pengguna lama Bank Jago Tapi Notif pesan ...,2,2026-05-27T01:34:25,Pengguna Google
3,fc0ec13d-a2bd-4ffc-8659-180aecfc231a,knp terjadi masalah trs seperti transfer gagal...,1,2026-05-27T01:27:58,Pengguna Google
4,7a45a9e7-d0f0-4a35-963d-7b66ab88f4a4,tolong benerin aplikasi nya ya dari tadi trans...,1,2026-05-27T00:38:59,Pengguna Google


## Verifikasi Output

In [16]:
# Verifikasi file exists dan valid
assert os.path.exists(OUTPUT_FILE), f'File {OUTPUT_FILE} tidak ditemukan!'
assert len(df) == len(sample), 'Jumlah baris CSV tidak sesuai dengan sample!'
print(f'OK: {OUTPUT_FILE} valid, {len(df)} rows')

OK: ../data/raw/reviews.csv valid, 10000 rows
